# 01 — Study area & AOI

**Purpose:** define the study area boundary once, save it to disk, and stop here.
Every later notebook clips its own data to this boundary as its *first* step —
this is the single biggest lever for keeping RAM usage low throughout the project.

**Study area:** Kandy, Kegalle, Nuwara Eliya, Ratnapura districts (central highlands,
the districts named in NBRO's Landslide Early Warning bulletins as active risk areas).

**Output:** `data/processed/aoi.gpkg` — a small GeoPackage, safe to reopen in every
subsequent notebook without ever loading a full raster or country-scale dataset.

**Before running:** get district (and ideally DS division) boundaries into
`data/raw/`. Options:
- Sri Lanka Survey Department (official, may require request)
- HDX / OCHA Common Operational Datasets — Admin Boundaries (`data.humdata.org`,
  search "Sri Lanka administrative boundaries")

Log whichever source you use in `docs/data_sources.md`.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

sys.path.append("../src")
from landslide_lk import geo_utils

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## Load admin boundaries

Point this at whatever boundary file you've placed in `data/raw/`. Adjust the
filename and the district-name column to match your actual source file — these
vary between Survey Department exports and HDX exports.

In [ ]:
# Adjust to match your actual downloaded file
boundaries_path = DATA_RAW / "lka_admin_boundaries.gpkg"

# admin = gpd.read_file(boundaries_path, layer="ADM2")  # district level
# admin.head()

# Placeholder so the notebook runs end-to-end before you've sourced real data:
# comment this out once boundaries_path exists.
print(f"Expected boundaries at: {boundaries_path.resolve()}")
print(f"Exists: {boundaries_path.exists()}")

## Filter to the four target districts

Adjust the column name (`ADM2_EN`, `DISTRICT_N`, `district`, etc. depending on
your source) and the exact district name spellings to match your file.

In [ ]:
TARGET_DISTRICTS = ["Kandy", "Kegalle", "Nuwara Eliya", "Ratnapura"]
DISTRICT_COLUMN = "ADM2_EN"  # <- adjust to match your source file's column name

# aoi = admin[admin[DISTRICT_COLUMN].isin(TARGET_DISTRICTS)].copy()
# assert len(aoi) == len(TARGET_DISTRICTS), (
#     f"Expected {len(TARGET_DISTRICTS)} districts, matched {len(aoi)} — "
#     "check district name spelling against your source file."
# )
# aoi

## Reproject and save

Standardize to UTM zone 44N (EPSG:32644) — Sri Lanka's local metric CRS — so
every later distance/area calculation (distance-to-river, slope, etc.) is in
real-world meters rather than degrees. This also matches the reprojection step
in prior Sri Lankan landslide studies for consistency.

In [ ]:
TARGET_CRS = "EPSG:32644"

# aoi_utm = aoi.to_crs(TARGET_CRS)
# aoi_dissolved = aoi_utm.dissolve()  # single polygon — used by clip_raster_to_aoi later

# out_path = DATA_PROCESSED / "aoi.gpkg"
# aoi_utm.to_file(out_path, layer="districts", driver="GPKG")
# aoi_dissolved.to_file(out_path, layer="dissolved", driver="GPKG")
# print(f"Saved: {out_path.resolve()}")

## Quick visual sanity check

Always look at the AOI before moving on — a wrong CRS or a mis-spelled district
name is much cheaper to catch here than three notebooks downstream.

In [ ]:
# fig, ax = plt.subplots(figsize=(6, 6))
# aoi_utm.plot(ax=ax, edgecolor="black", column=DISTRICT_COLUMN, legend=True)
# ax.set_title("Study area: " + ", ".join(TARGET_DISTRICTS))
# plt.show()

## Next

Close this notebook — restart the kernel or just move on, no need to keep it
open. `notebooks/02_terrain.ipynb` will load only `data/processed/aoi.gpkg`
(a few KB) and won't touch anything from this session's memory.

Every later data-layer notebook (`02_terrain`, `03_climate`, `04_context`,
`05_labels`) should start by loading this AOI and clipping its own raw source
data to it via `geo_utils.clip_raster_to_aoi()` before doing anything else.